# Model Development

This notebook focuses on developing and training machine learning models for predicting term deposit subscriptions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path
import sys

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from src.data_ingestion.data_ingestion import load_data
from src.data_preprocessing.data_preprocessing import DataPreprocessor
from src.feature_engineering.feature_engineering import FeatureEngineer

In [ ]:
# Load data
data_path = '../data/raw/bank.csv'
df = load_data(data_path)
print(f"Dataset shape: {df.shape}")

In [ ]:
# Split features and target
target_column = 'y'
X = df.drop(columns=[target_column])
y = df[target_column]

# Encode target variable
y = y.map({'yes': 1, 'no': 0})

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Identify column types
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numeric_cols}")

In [ ]:
# Preprocess training data
X_train_processed = preprocessor.handle_missing_values(X_train, strategy='mean')
X_train_processed = preprocessor.encode_categorical(X_train_processed, categorical_cols)
X_train_processed = preprocessor.scale_features(X_train_processed, numeric_cols)

print(f"Processed training data shape: {X_train_processed.shape}")

In [ ]:
# Preprocess test data (using fitted scalers/encoders)
X_test_processed = preprocessor.handle_missing_values(X_test, strategy='mean')
X_test_processed = preprocessor.encode_categorical(X_test_processed, categorical_cols)
X_test_processed = preprocessor.scale_features(X_test_processed, numeric_cols, fit=False)

print(f"Processed test data shape: {X_test_processed.shape}")

In [ ]:
# Feature engineering (optional)
feature_engineer = FeatureEngineer()

# Example: Create interaction features
# feature_pairs = [('age', 'balance'), ('duration', 'campaign')]
# X_train_processed = feature_engineer.create_interaction_features(X_train_processed, feature_pairs)
# X_test_processed = feature_engineer.create_interaction_features(X_test_processed, feature_pairs)

print("Feature engineering complete")

In [ ]:
# Import MLflow and training functions
import mlflow
mlflow.set_tracking_uri("../mlruns")

from src.models.train_model import train_model, hyperparameter_tuning

In [ ]:
# Train Random Forest model
rf_params = {
    'n_estimators': 200,
    'max_depth': 20,
    'min_samples_split': 5,
    'random_state': 42
}

rf_result = train_model(
    X_train_processed, y_train,
    model_type='random_forest',
    params=rf_params,
    experiment_name='term_deposit_prediction'
)

print(f"Random Forest CV F1: {rf_result['cv_f1_mean']:.4f} (+/- {rf_result['cv_f1_std']:.4f})")

In [ ]:
# Train Gradient Boosting model
gb_params = {
    'n_estimators': 200,
    'learning_rate': 0.1,
    'max_depth': 5,
    'random_state': 42
}

gb_result = train_model(
    X_train_processed, y_train,
    model_type='gradient_boosting',
    params=gb_params,
    experiment_name='term_deposit_prediction'
)

print(f"Gradient Boosting CV F1: {gb_result['cv_f1_mean']:.4f} (+/- {gb_result['cv_f1_std']:.4f})")

In [ ]:
# Train Logistic Regression model
lr_params = {
    'C': 1.0,
    'penalty': 'l2',
    'random_state': 42
}

lr_result = train_model(
    X_train_processed, y_train,
    model_type='logistic_regression',
    params=lr_params,
    experiment_name='term_deposit_prediction'
)

print(f"Logistic Regression CV F1: {lr_result['cv_f1_mean']:.4f} (+/- {lr_result['cv_f1_std']:.4f})")

In [ ]:
# Hyperparameter tuning for best model
tuning_result = hyperparameter_tuning(
    X_train_processed, y_train,
    model_type='random_forest'
)

print(f"Best parameters: {tuning_result['best_params']}")
print(f"Best CV score: {tuning_result['best_score']:.4f}")

## Model Summary

Compare model performance and select the best model for deployment.